# Stage 2 nvBench post-query preparation

Mount Drive, install the repository package, prepare nvBench as materialized post-query examples, and verify output artifacts.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import stat
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit

from google.colab import drive

try:
    from google.colab import userdata
except Exception:  # pragma: no cover - Colab only
    userdata = None

SENSITIVE_VALUES: list[str] = []


def read_secret(name: str, token_files: list[Path] | None = None) -> str | None:
    value = os.environ.get(name)
    if value:
        return value.strip()
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            return value.strip()
    for token_file in token_files or []:
        if token_file.exists():
            value = token_file.read_text(encoding='utf-8').strip()
            if value:
                return value
    return None


def scrub(text: str) -> str:
    result = text
    for value in SENSITIVE_VALUES:
        if value:
            result = result.replace(value, '***')
    return result


def scrub_cmd(cmd: list[str]) -> str:
    return ' '.join(scrub(str(part)) for part in cmd)


def run(cmd: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    print('RUN', scrub_cmd(cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(scrub(result.stdout))
    if result.stderr:
        print(scrub(result.stderr))
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {scrub_cmd(cmd)}')
    return result


def configure_github_auth(token_files: list[Path]) -> tuple[str, str | None]:
    token = read_secret('GITHUB_TOKEN', token_files) or read_secret('GH_TOKEN', token_files)
    if not token:
        return 'anonymous_https', None
    SENSITIVE_VALUES.append(token)
    netrc_path = Path.home() / '.netrc'
    netrc_path.write_text(
        'machine github.com\n'
        '  login x-access-token\n'
        f'  password {token}\n',
        encoding='utf-8',
    )
    os.chmod(netrc_path, stat.S_IRUSR | stat.S_IWUSR)
    return 'token_https_url_redacted', token


def authenticated_repo_url(repo_url: str, token: str | None) -> str:
    if not token or not repo_url.startswith('https://'):
        return repo_url
    parts = urlsplit(repo_url)
    if parts.hostname != 'github.com':
        return repo_url
    netloc = f'x-access-token:{quote(token, safe="")}@{parts.netloc}'
    return urlunsplit((parts.scheme, netloc, parts.path, parts.query, parts.fragment))


def is_disposable_content_checkout(path: Path) -> bool:
    resolved = path.resolve()
    return str(resolved).startswith('/content/') and not str(resolved).startswith('/content/drive/')


def clone_repo(repo_root: Path, repo_url: str, auth_repo_url: str, branch: str) -> str:
    if repo_root.exists():
        if not is_disposable_content_checkout(repo_root):
            raise RuntimeError(f'Refusing to remove non-disposable checkout path: {repo_root}')
        shutil.rmtree(repo_root)
    repo_root.parent.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', '--branch', branch, '--single-branch', auth_repo_url, str(repo_root)])
    run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=repo_root)
    run(['git', 'rev-parse', 'HEAD'], cwd=repo_root)
    return 'cloned_branch'


def update_repo(repo_root: Path, repo_url: str, auth_repo_url: str, branch: str) -> str:
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Repo target exists but is not a git checkout: {repo_root}')
    run(['git', 'remote', 'set-url', 'origin', auth_repo_url], cwd=repo_root)
    try:
        run(['git', 'fetch', 'origin', branch], cwd=repo_root)
        run(['git', 'checkout', branch], cwd=repo_root)
        run(['git', 'pull', '--ff-only', 'origin', branch], cwd=repo_root)
        run(['git', 'rev-parse', 'HEAD'], cwd=repo_root)
    finally:
        run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=repo_root, check=False)
    return 'pulled_existing_checkout'


def sync_git_repo(repo_root: Path, repo_url: str, auth_repo_url: str, branch: str) -> str:
    if not repo_root.exists():
        return clone_repo(repo_root, repo_url, auth_repo_url, branch)
    if (repo_root / '.git').exists():
        return update_repo(repo_root, repo_url, auth_repo_url, branch)
    if any(repo_root.iterdir()):
        raise RuntimeError(f'Repo target exists but is not a git checkout: {repo_root}')
    return clone_repo(repo_root, repo_url, auth_repo_url, branch)


drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path(os.environ.get('T2V_DRIVE_ROOT', '/content/drive/MyDrive/diploma/petr_text_to_visualization_part'))
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

REPO_URL = os.environ.get('T2V_REPO_URL', 'https://github.com/petrussia/NL2BI-AI-assistant.git')
REPO_BRANCH = os.environ.get('T2V_REPO_BRANCH', 'experiments/peter')
DEFAULT_REPO_ROOT = Path(os.environ.get('T2V_REPO_ROOT', '/content/petukhov_t2v_repo'))
TOKEN_FILES = [
    Path(os.environ['T2V_GITHUB_TOKEN_FILE']) if os.environ.get('T2V_GITHUB_TOKEN_FILE') else None,
    DRIVE_ROOT / 'secrets' / 'github_token.txt',
]
TOKEN_FILES = [path for path in TOKEN_FILES if path is not None]

auth_mode, github_token = configure_github_auth(TOKEN_FILES)
if not github_token and os.environ.get('T2V_ALLOW_ANONYMOUS_GITHUB', '0') != '1':
    print('STAGE2_SETUP_BLOCKED')
    print(json.dumps({
        'reason': 'GITHUB_TOKEN or GH_TOKEN is not available to this Colab notebook via google.colab.userdata, environment, or Drive token file',
        'repo_url': REPO_URL,
        'branch': REPO_BRANCH,
        'hint': 'Add a Colab Secret named GITHUB_TOKEN and enable notebook access, or create /content/drive/MyDrive/diploma/petr_text_to_visualization_part/secrets/github_token.txt containing only the token. Do not hardcode the token in a notebook cell.',
        'token_files_checked': [str(path) for path in TOKEN_FILES],
    }, ensure_ascii=False, indent=2))
    raise RuntimeError('GITHUB_TOKEN is required for private repository sync')

auth_repo_url = authenticated_repo_url(REPO_URL, github_token)
repo_root = DEFAULT_REPO_ROOT
sync_status = sync_git_repo(repo_root, REPO_URL, auth_repo_url, REPO_BRANCH)

if not (repo_root / 'pyproject.toml').exists():
    print('STAGE2_SETUP_BLOCKED')
    print(json.dumps({
        'reason': 'Colab-visible repository checkout was synchronized but pyproject.toml root marker is missing',
        'repo_root': str(repo_root),
        'repo_url': REPO_URL,
        'branch': REPO_BRANCH,
        'sync_status': sync_status,
    }, ensure_ascii=False, indent=2))
    raise RuntimeError('T2V_REPO_ROOT is not an installable repository checkout')

run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo_root / 'requirements.txt')])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_root)])
os.environ['T2V_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['T2V_REPO_ROOT'] = str(repo_root)
os.environ['T2V_REPO_BRANCH'] = REPO_BRANCH
version_result = run([sys.executable, '-c', 'import t2v_eval; print(t2v_eval.__version__)'])
print('STAGE2_SETUP_OK')
print(json.dumps({
    'drive_root': str(DRIVE_ROOT),
    'repo_root': str(repo_root),
    'repo_url': REPO_URL,
    'branch': REPO_BRANCH,
    'sync_status': sync_status,
    'github_auth': auth_mode,
    't2v_eval_version': version_result.stdout.strip(),
}, ensure_ascii=False, indent=2))


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path(os.environ['T2V_REPO_ROOT'])
drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'prepare_nvbench.py'),
    '--drive-root', str(drive_root),
    '--sample-size', '20',
    '--min-successful', '1',
    '--json',
]
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'sample20 preparation failed with exit code {result.returncode}')
print('STAGE2_SAMPLE20_OK')


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

repo_root = Path(os.environ['T2V_REPO_ROOT'])
drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'prepare_nvbench.py'),
    '--drive-root', str(drive_root),
    '--sample-size', '200',
    '--min-successful', '50',
    '--json',
]
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'sample200 preparation failed with exit code {result.returncode}')
print('STAGE2_SAMPLE200_OK')


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
processed = drive_root / 'datasets' / 'processed' / 'nvbench_postquery'
required = [
    processed / 'examples.jsonl',
    processed / 'examples_sample200.jsonl',
    processed / 'dataset_card.md',
    processed / 'prepare_result.json',
    processed / 'tables',
]
missing = [str(path) for path in required if not path.exists()]
summary = json.loads((processed / 'prepare_result.json').read_text(encoding='utf-8')) if (processed / 'prepare_result.json').exists() else {}
table_count = len(list((processed / 'tables').glob('*.csv'))) if (processed / 'tables').exists() else 0
print(json.dumps({'processed': str(processed), 'missing': missing, 'summary': summary, 'table_count': table_count}, ensure_ascii=False, indent=2))
if missing:
    raise RuntimeError(f'Missing Stage 2 artifacts: {missing}')
if summary.get('successful_examples', 0) < 50:
    raise RuntimeError('Less than 50 successful examples after sample200 preparation')
print('STAGE2_ARTIFACTS_OK')
